# PC General

薄入口 notebook：只负责导入两个 cell、配置开关、添加刺激与 probe、运行和分析。

In [1]:
from pathlib import Path
import sys
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import brainunit as u
from neuron import h

def find_repo_root(start=None):
    cwd = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "braincell").exists() and (candidate / "examples").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from examples.neuron_compare.cell.pc_ma2024.debug.pc_parameters import (
    DEFAULT_MORPH_PATH,
    DEFAULT_NRNMECH_PATH,
    PCConfig,
    PCToggles,
    load_pc24_params,
)
from examples.neuron_compare.cell.pc_ma2024.debug.pc_neuron_debug import PC as NeuronPC
from examples.neuron_compare.cell.pc_ma2024.debug.pc_braincell_debug import PC as BrainCellPC

import braincell
from braincell import mech
from braincell.filter import at
print("braincell import:", braincell.__file__)
import brainstate
brainstate.environ.set(precision=64)

--No graphics will be displayed.


braincell import: /home/swl/braincell/braincell/__init__.py


In [2]:
params = load_pc24_params()
toggles = PCToggles(
    leak=True,
    nav=False,
    kv1p1=False,
    kv1p5=False,
    kv3p3=False,
    kv3p4=False,
    kv4p3=False,
    kir2p3=False,
    kca1p1=False,
    kca2p2=False,
    kca3p1=False,
    cav21=False,
    cav31=False,
    cav32=False,
    cav33=False,
    hcn1=False,
    cdp=False,
)
config = PCConfig(toggles=toggles, temperature_celsius=36.0, v_init_mV=-65.0)

dt_ms = 0.1
duration_ms = 100.0
delay_ms = 10.0
stim_dur_ms = 80
amp_nA = 0.5

# Centralized soma diagnostics for cav31-only debugging.
soma_diag_config = {
    "ica": True,
    "cai": True,
    "cao": True,
    "gates": True,
}


In [3]:
neuron_pc = NeuronPC(DEFAULT_MORPH_PATH, params=params, config=config, nrnmech_path=DEFAULT_NRNMECH_PATH).build()
braincell_pc = BrainCellPC(DEFAULT_MORPH_PATH, params=params, config=config).build()

display(pd.Series(neuron_pc.summary(), dtype=object))
display(pd.Series(braincell_pc.summary(), dtype=object))
display(neuron_pc.branch_table().head())
display(braincell_pc.branch_table().head())



3856 lines read


/home/swl/braincell/braincell/io/asc/reader.py:584: UserWarning: from_points() produced 1 zero-length segment(s) from coincident consecutive points at index pair(s) [6]. These degenerate segments are kept but contribute zero volume.
  return branch_class_for_type(segment.branch_type).from_points(
/home/swl/braincell/braincell/io/asc/reader.py:584: UserWarning: from_points() produced 1 zero-length segment(s) from coincident consecutive points at index pair(s) [8]. These degenerate segments are kept but contribute zero volume.
  return branch_class_for_type(segment.branch_type).from_points(
/home/swl/braincell/braincell/io/asc/reader.py:584: UserWarning: from_points() produced 1 zero-length segment(s) from coincident consecutive points at index pair(s) [4]. These degenerate segments are kept but contribute zero volume.
  return branch_class_for_type(segment.branch_type).from_points(
/home/swl/braincell/braincell/io/asc/reader.py:584: UserWarning: from_points() produced 1 zero-length segm

backend                                                          neuron
morph_path            /home/swl/braincell/examples/neuron_compare/Ce...
indiv                                                               138
toggles               {'leak': True, 'nav': False, 'kv1p1': False, '...
branch_counts              {'n_soma': 1, 'n_dend': 460, 'n_total': 461}
compartment_counts                                {'n_total_nseg': 465}
threshold_hits                    {'n_thick_dend': 26, 'n_nav_dend': 1}
enabled_mechanisms    {'soma': ['leak'], 'dend_all': ['leak'], 'dend...
ion_status            {'ena_enabled': False, 'ek_enabled': False, 'e...
dtype: object

backend                                                       braincell
morph_path            /home/swl/braincell/examples/neuron_compare/Ce...
indiv                                                               138
toggles               {'leak': True, 'nav': False, 'kv1p1': False, '...
branch_counts              {'n_soma': 1, 'n_dend': 460, 'n_total': 461}
compartment_counts                                  {'n_total_cv': 465}
threshold_hits                    {'n_thick_dend': 26, 'n_nav_dend': 1}
enabled_mechanisms    {'soma': ['leak'], 'dend_all': ['leak'], 'dend...
ion_status                          {'na': None, 'k': None, 'ca': None}
dtype: object

,branch_index,branch_name,branch_type,diam_um,diam_arc_mean_um,cm_uF_cm2,nseg,is_thick_dend,is_nav_dend,has_cav21,...,has_kca3p1,has_kir2p3,has_kv1p1,has_kv1p5,has_kv3p3,has_kv3p4,has_kv4p3,has_leak,has_nav,enabled_mechanisms
0,0,soma[0],soma,15.744254,15.744254,2.000000,1,False,False,False,...,False,False,False,False,False,False,False,True,False,[leak]
1,1,dend[0],dendrite,3.626054,3.626054,2.000000,1,True,True,False,...,False,False,False,False,False,False,False,True,False,[leak]
2,2,dend[1],dendrite,1.900000,1.900000,2.000000,1,True,False,False,...,False,False,False,False,False,False,False,True,False,[leak]
3,3,dend[2],dendrite,1.900000,1.900000,2.000000,1,True,False,False,...,False,False,False,False,False,False,False,True,False,[leak]
4,4,dend[3],dendrite,1.342487,1.342487,3.934183,1,False,False,False,...,False,False,False,False,False,False,False,True,False,[leak]


,branch_index,branch_name,branch_type,diam_arc_mean_um,cm_uF_cm2,n_cv,is_thick_dend,is_nav_dend,has_cav21,has_cav31,...,has_kca3p1,has_kir2p3,has_kv1p1,has_kv1p5,has_kv3p3,has_kv3p4,has_kv4p3,has_leak,has_nav,enabled_mechanisms
0,0,soma,soma,15.744263,2.000000,1,False,False,False,False,...,False,False,False,False,False,False,False,True,False,[leak]
1,1,dendrite_0,dendrite,3.626055,2.000000,1,True,True,False,False,...,False,False,False,False,False,False,False,True,False,[leak]
2,2,dendrite_1,dendrite,1.900000,2.000000,1,True,False,False,False,...,False,False,False,False,False,False,False,True,False,[leak]
3,3,dendrite_2,dendrite,1.900000,2.000000,1,True,False,False,False,...,False,False,False,False,False,False,False,True,False,[leak]
4,4,dendrite_3,dendrite,1.342488,3.934181,1,False,False,False,False,...,False,False,False,False,False,False,False,True,False,[leak]


In [4]:
nrn_voltage_probes = neuron_pc.attach_voltage_probes(all_compartments=True, soma=True)
bc_voltage_probes = braincell_pc.attach_voltage_probes(all_compartments=True, soma=True)

def attach_cav31_soma_diagnostics(neuron_pc, braincell_pc, enabled):
    handles = {}
    nrn_seg = neuron_pc.root_soma(0.5)
    if enabled.get("ica", False):
        handles["ica_neuron"] = h.Vector().record(nrn_seg.Cav3p1_MA24_PC._ref_ica)
        braincell_pc.cell.place(
            at("soma", 0.5),
            mech.CurrentProbe(ion="ca", mechanism="Cav3p1_MA2024_PC_Frozen"),
        )
        handles["ica_braincell_trace"] = "soma(0.5)_Cav3p1_MA2024_PC_Frozen_current"
    if enabled.get("cai", False):
        handles["cai_neuron"] = h.Vector().record(nrn_seg._ref_cai)
        braincell_pc.cell.place(at("soma", 0.5), mech.MechanismProbe(mechanism="ca", field="Ci"))
        handles["cai_braincell_trace"] = "soma(0.5)_ca_Ci"
    if enabled.get("cao", False):
        handles["cao_neuron"] = h.Vector().record(nrn_seg._ref_cao)
        braincell_pc.cell.place(at("soma", 0.5), mech.MechanismProbe(mechanism="ca", field="Co"))
        handles["cao_braincell_trace"] = "soma(0.5)_ca_Co"
    if enabled.get("gates", False):
        handles["p_neuron"] = h.Vector().record(nrn_seg.Cav3p1_MA24_PC._ref_m)
        handles["q_neuron"] = h.Vector().record(nrn_seg.Cav3p1_MA24_PC._ref_h)
        braincell_pc.cell.place(at("soma", 0.5), mech.MechanismProbe(mechanism="Cav3p1_MA2024_PC_Frozen", field="p"))
        braincell_pc.cell.place(at("soma", 0.5), mech.MechanismProbe(mechanism="Cav3p1_MA2024_PC_Frozen", field="q"))
        handles["p_braincell_trace"] = "soma(0.5)_Cav3p1_MA2024_PC_Frozen_p"
        handles["q_braincell_trace"] = "soma(0.5)_Cav3p1_MA2024_PC_Frozen_q"
    return handles

soma_diag_handles = attach_cav31_soma_diagnostics(neuron_pc, braincell_pc, soma_diag_config)


AttributeError: Cav3p1_MA24_PC, the mechanism does not exist at soma[0](0.5)

In [ ]:
stim = h.IClamp(neuron_pc.root_soma(0.5))
stim.delay = delay_ms
stim.dur = stim_dur_ms
stim.amp = amp_nA
h.cvode_active(0)
h.dt = dt_ms
h.steps_per_ms = 1.0 / h.dt
h.celsius = config.temperature_celsius
h.tstop = duration_ms
h.v_init = config.v_init_mV
t_neuron = h.Vector().record(h._ref_t)
h.finitialize(h.v_init)
h.run()

braincell_pc.cell.place(
    at("soma", 0.5),
    mech.CurrentClamp.step(amp_nA * u.nA, stim_dur_ms * u.ms, delay=delay_ms * u.ms),
)
braincell_pc.cell.init_state()
nrn_seg = neuron_pc.root_soma(0.5)
bc_ca = braincell_pc.cell.get_ion("ca") if braincell_pc.summary()["ion_status"]["ca"] is not None else None
if bc_ca is not None:
    ca_diagnostic = {
        "neuron_eca_mV": float(nrn_seg.eca) if hasattr(nrn_seg, "eca") else None,
        "neuron_cai_mM": float(nrn_seg.cai) if hasattr(nrn_seg, "cai") else None,
        "neuron_cao_mM": float(nrn_seg.cao) if hasattr(nrn_seg, "cao") else None,
        "braincell_ca_type": type(bc_ca).__name__,
        "braincell_E_mV": float(np.asarray(bc_ca.E.to_decimal(u.mV), dtype=float).reshape(-1)[0]),
        "braincell_Ci_mM": float(np.asarray(bc_ca.Ci.value.to_decimal(u.mM), dtype=float).reshape(-1)[0]),
        "braincell_Co_mM": float(np.asarray(bc_ca.Co.to_decimal(u.mM), dtype=float).reshape(-1)[0]),
    }
    display(ca_diagnostic)
braincell_pc.cell.reset_state()
bc_run = braincell_pc.cell.run(dt=dt_ms * u.ms, duration=duration_ms * u.ms)


In [ ]:
nrn_v = neuron_pc.collect_voltage_results(nrn_voltage_probes)
bc_v = braincell_pc.collect_voltage_results(bc_voltage_probes, bc_run)

reference_time_ms = np.round(np.arange(0.0, duration_ms, dt_ms, dtype=float), decimals=12)
neuron_time_raw = np.asarray(t_neuron, dtype=float).reshape(-1)

def trim_neuron_trace(values, reference_time_ms):
    values = np.asarray(values, dtype=float).reshape(-1)
    if values.shape[0] == reference_time_ms.shape[0] + 1:
        return values[1:]
    return values

def trim_current_pair(neuron_values, braincell_values):
    neuron_values = np.asarray(neuron_values, dtype=float).reshape(-1)
    braincell_values = np.asarray(braincell_values, dtype=float).reshape(-1)
    neuron_values = neuron_values[2:]
    braincell_values = braincell_values[:-1]
    n = min(neuron_values.shape[0], braincell_values.shape[0])
    return neuron_values[:n], braincell_values[:n]

neuron_soma = trim_neuron_trace(nrn_v["soma_voltage_mV"], reference_time_ms)
if nrn_v["compartment_voltage_mV"].shape[0] == reference_time_ms.shape[0] + 1:
    neuron_comp = nrn_v["compartment_voltage_mV"][1:, :]
else:
    neuron_comp = nrn_v["compartment_voltage_mV"]
braincell_soma = bc_v["soma_voltage_mV"]
braincell_comp = bc_v["compartment_voltage_mV"]

soma_diag = {}
if soma_diag_config.get("ica", False):
    neuron_ica_raw = -np.asarray(soma_diag_handles["ica_neuron"], dtype=float).reshape(-1)
    braincell_ica_raw = np.asarray(bc_run.traces[soma_diag_handles["ica_braincell_trace"]].to_decimal(u.mA / (u.cm ** 2)), dtype=float).reshape(-1)
    neuron_ica_mA_cm2, braincell_ica_mA_cm2 = trim_current_pair(neuron_ica_raw, braincell_ica_raw)
    current_time_ms = reference_time_ms[:braincell_ica_mA_cm2.shape[0]]
    soma_diag["ica"] = (neuron_ica_mA_cm2, braincell_ica_mA_cm2, "mA/cm²", current_time_ms)
if soma_diag_config.get("cai", False):
    neuron_cai_mM = trim_neuron_trace(soma_diag_handles["cai_neuron"], reference_time_ms)
    braincell_cai_mM = np.asarray(bc_run.traces[soma_diag_handles["cai_braincell_trace"]].to_decimal(u.mM), dtype=float).reshape(-1)
    soma_diag["cai"] = (neuron_cai_mM, braincell_cai_mM, "mM")
if soma_diag_config.get("cao", False):
    neuron_cao_mM = trim_neuron_trace(soma_diag_handles["cao_neuron"], reference_time_ms)
    braincell_cao_mM = np.asarray(bc_run.traces[soma_diag_handles["cao_braincell_trace"]].to_decimal(u.mM), dtype=float).reshape(-1)
    soma_diag["cao"] = (neuron_cao_mM, braincell_cao_mM, "mM")
if soma_diag_config.get("gates", False):
    neuron_p = trim_neuron_trace(soma_diag_handles["p_neuron"], reference_time_ms)
    neuron_q = trim_neuron_trace(soma_diag_handles["q_neuron"], reference_time_ms)
    braincell_p = np.asarray(bc_run.traces[soma_diag_handles["p_braincell_trace"]], dtype=float).reshape(-1)
    braincell_q = np.asarray(bc_run.traces[soma_diag_handles["q_braincell_trace"]], dtype=float).reshape(-1)
    soma_diag["p"] = (neuron_p, braincell_p, "")
    soma_diag["q"] = (neuron_q, braincell_q, "")

def metric_dict(a, b):
    delta = np.asarray(b) - np.asarray(a)
    return {
        "rmse": float(np.sqrt(np.mean(delta ** 2))),
        "max_abs": float(np.max(np.abs(delta))),
        "mean_abs": float(np.mean(np.abs(delta))),
    }

delta_soma = braincell_soma - neuron_soma
soma_summary = {
    "voltage_rmse_mV": float(np.sqrt(np.mean(delta_soma ** 2))),
    "voltage_max_abs_mV": float(np.max(np.abs(delta_soma))),
    "voltage_mean_abs_mV": float(np.mean(np.abs(delta_soma))),
}
for key, values in soma_diag.items():
    neuron_values, braincell_values, unit_label = values[:3]
    stats = metric_dict(neuron_values, braincell_values)
    suffix = unit_label if unit_label else ""
    soma_summary[f"{key}_rmse_{suffix}"] = stats["rmse"]
    soma_summary[f"{key}_max_abs_{suffix}"] = stats["max_abs"]
    soma_summary[f"{key}_mean_abs_{suffix}"] = stats["mean_abs"]

soma_summary


In [ ]:
pair_table = pd.merge(
    bc_v["compartment_table"],
    nrn_v["compartment_table"],
    on=["branch_index", "branch_type", "local_index"],
    suffixes=("_braincell", "_neuron"),
)
metric_rows = []
for row in pair_table.itertuples(index=False):
    bc_idx = int(row.compartment_index_braincell)
    nrn_idx = int(row.compartment_index_neuron)
    delta = braincell_comp[:, bc_idx] - neuron_comp[:, nrn_idx]
    metric_rows.append(
        {
            "branch_index": int(row.branch_index),
            "branch_name": row.branch_name_braincell,
            "local_index": int(row.local_index),
            "mean_abs_mV": float(np.mean(np.abs(delta))),
            "rmse_mV": float(np.sqrt(np.mean(delta ** 2))),
            "max_abs_mV": float(np.max(np.abs(delta))),
        }
    )
compartment_metrics = pd.DataFrame(metric_rows)
display(compartment_metrics.sort_values("max_abs_mV", ascending=False).head(20))
display(
    {
        "mean_of_mean_abs_mV": float(compartment_metrics["mean_abs_mV"].mean()),
        "mean_of_rmse_mV": float(compartment_metrics["rmse_mV"].mean()),
        "max_abs_mV": float(compartment_metrics["max_abs_mV"].max()),
    }
)

In [ ]:
# Manually tune these values for presentation or manuscript export.
figure_style = {
    "fig_width": 7.2,
    "panel_height": 1.9,
    "dpi": 180,
    "font_family": "DejaVu Sans",
    "title_size": 9.5,
    "axis_label_size": 8.5,
    "tick_label_size": 7.5,
    "legend_size": 7.5,
    "line_width": 1.35,
    "spine_width": 0.85,
    "tick_width": 0.75,
    "tick_length": 3.0,
    "grid_alpha": 0.16,
    "neuron_color": "#1F4E79",
    "braincell_color": "#B03A2E",
    "save_path": None,  # e.g. "pc_debug_soma_comparison.pdf"
}

plot_labels = {
    "voltage": "Membrane voltage",
    "ica": r"$I_{CaV3.1}$",
    "cai": r"$[Ca^{2+}]_i$",
    "cao": r"$[Ca^{2+}]_o$",
    "p": r"CaV3.1 activation $p$",
    "q": r"CaV3.1 inactivation $q$",
}

plot_specs = [("voltage", reference_time_ms, neuron_soma, braincell_soma, "Voltage (mV)")]
for key in ["ica", "cai", "cao", "p", "q"]:
    if key in soma_diag:
        neuron_values, braincell_values, unit_label = soma_diag[key][:3]
        time_ms = soma_diag[key][3] if len(soma_diag[key]) > 3 else reference_time_ms
        ylabel = f"{plot_labels[key]} ({unit_label})" if unit_label else plot_labels[key]
        plot_specs.append((key, time_ms, neuron_values, braincell_values, ylabel))

with plt.rc_context(
    {
        "font.family": figure_style["font_family"],
        "font.size": figure_style["tick_label_size"],
        "axes.linewidth": figure_style["spine_width"],
        "axes.labelsize": figure_style["axis_label_size"],
        "axes.titlesize": figure_style["title_size"],
        "xtick.labelsize": figure_style["tick_label_size"],
        "ytick.labelsize": figure_style["tick_label_size"],
        "legend.fontsize": figure_style["legend_size"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }
):
    fig, axes = plt.subplots(
        len(plot_specs),
        1,
        figsize=(figure_style["fig_width"], figure_style["panel_height"] * len(plot_specs)),
        dpi=figure_style["dpi"],
        sharex=True,
        constrained_layout=True,
    )
    if len(plot_specs) == 1:
        axes = [axes]

    for ax, (name, time_ms, neuron_values, braincell_values, ylabel) in zip(axes, plot_specs):
        ax.plot(
            time_ms,
            neuron_values,
            color=figure_style["neuron_color"],
            linewidth=figure_style["line_width"],
            label="NEURON",
        )
        ax.plot(
            time_ms,
            braincell_values,
            color=figure_style["braincell_color"],
            linewidth=figure_style["line_width"],
            linestyle="--",
            dashes=(3.2, 1.8),
            label="BrainCell",
        )
        ax.set_ylabel(ylabel)
        ax.set_title(plot_labels.get(name, name), loc="left", pad=3)
        ax.grid(axis="y", color="0.15", alpha=figure_style["grid_alpha"], linewidth=0.55)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(
            axis="both",
            which="major",
            width=figure_style["tick_width"],
            length=figure_style["tick_length"],
            direction="out",
        )
        ax.margins(x=0)

    legend = axes[0].legend(
        frameon=True,
        loc="best",
        handlelength=2.6,
        borderaxespad=0.25,
        borderpad=0.35,
        labelspacing=0.25,
        framealpha=0.92,
        facecolor="white",
        edgecolor="0.35",
    )
    legend.get_frame().set_linewidth(0.65)
    legend.set_zorder(10)
    axes[-1].set_xlabel("Time (ms)")

    if figure_style["save_path"]:
        fig.savefig(figure_style["save_path"], bbox_inches="tight", dpi=figure_style["dpi"])
    plt.show()
